# ทดสอบ Clef (NVFP4) API บน Vast.ai RTX 5090

Server: `server/server.py` (vLLM pooling + joint head, NVFP4 kernel จริงบน Blackwell) — endpoint เดียวกับ Jev/SystemOne `POST /v1/systemone`

| คำถาม `type` | ได้อะไรกลับ |
|:--|:--|
| `noul` | ความน่าจะเป็นที่คำตอบคือ "ใช่" (0–1) |
| `choice` | ตัวเลือกที่ดีที่สุด + ความน่าจะเป็นของทุกตัวเลือก |
| `score` | ค่าคาดหวังบนสเกล 0..N-1 + การกระจาย |

**Direct URL จาก Vast:** notebook ถาม Vast API หา instance ตาม label (`CLEF_VAST_LABEL`, default `clef-5090`)
แล้วสร้าง URL `http://<public_ipaddr>:<HostPort ที่ map กับ 8000/tcp>` เอง — ไม่ต้องแก้ URL ทุกครั้งที่เช่าเครื่องใหม่

- `VAST_API_KEY` อ่านจาก `/root/workspace/DGX_2024/.env`
- `CLEF_API_KEY` (Bearer ของ server) อ่านจาก `.env` ในโฟลเดอร์นี้
- ตั้ง `CLEF_BASE_URL` ใน `.env` ได้ถ้าต้องการ override

In [3]:
import json, os, time, statistics, base64, io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import requests

def load_env(path):
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip().removeprefix("export "), v.split(" #")[0].strip().strip('"').strip("'"))

load_env(".env")                              # CLEF_API_KEY
load_env("/root/workspace/DGX_2024/.env")     # VAST_API_KEY
HEADERS = {"Authorization": f"Bearer {os.environ['CLEF_API_KEY']}"}

def vast_direct_url(label=os.environ.get("CLEF_VAST_LABEL", "clef-5090"), port="8000/tcp"):
    """หา instance ตาม label บน Vast แล้วคืน direct URL http://<public_ip>:<host port ของ 8000/tcp>"""
    r = requests.get("https://console.vast.ai/api/v0/instances/", params={"owner": "me"},
                     headers={"Authorization": f"Bearer {os.environ['VAST_API_KEY']}"}, timeout=30)
    r.raise_for_status()
    rows = [i for i in r.json()["instances"] if (i.get("label") or "").startswith(label)]
    if not rows:
        raise RuntimeError(f"ไม่พบ Vast instance label {label!r}")
    inst = max(rows, key=lambda i: i.get("start_date") or 0)
    hp = ((inst.get("ports") or {}).get(port) or [{}])[0].get("HostPort")
    if inst.get("actual_status") != "running" or not hp:
        raise RuntimeError(f"instance {inst['id']} = {inst.get('actual_status')!r} ยังไม่มี port {port}")
    print(f"vast instance {inst['id']} {inst['gpu_name']} {inst.get('geolocation')} ${inst['dph_total']:.3f}/hr")
    return f"http://{inst['public_ipaddr'].strip()}:{hp}"

BASE = (os.environ.get("CLEF_BASE_URL") or vast_direct_url()).rstrip("/")
print("server:", BASE)

RuntimeError: instance 53874538 = 'loading' ยังไม่มี port 8000/tcp

## 1. Health check

In [ ]:
r = requests.get(f"{BASE}/health", timeout=10)
print(r.status_code, r.json())

In [ ]:
def systemone(state, questions, images=None, timeout=60):
    body = {"model": "clef", "state": state, "questions": questions}
    if images:
        body["images"] = images
    t0 = time.perf_counter()
    r = requests.post(f"{BASE}/v1/systemone", json=body, headers=HEADERS, timeout=timeout)
    rtt = (time.perf_counter() - t0) * 1000
    r.raise_for_status()
    res = r.json()
    res["rtt_ms"] = round(rtt, 1)
    return res

def show(res):
    print(f"server {res.get('latency_ms')} ms | round-trip {res['rtt_ms']} ms | input_tokens {res['usage']['input_tokens']}")
    print(json.dumps(res["answers"], indent=2, ensure_ascii=False))

## 2. ตัวอย่างจาก model card — routing ticket (choice + score + noul)

In [ ]:
res = systemone(
    "Our checkout started returning errors and orders are blocked.",
    {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle the message?",
            "criteria": {"billing": "Payments or invoices", "technical": "Bugs or outages"},
        },
        "urgency": {"type": "score", "criteria": ["Can wait", "This week", "Today"]},
        "outage": {"type": "noul", "instructions": "Is a service down?"},
    },
)
show(res)

## 3. State เป็น JSON (ข้อมูลของแอป) + ภาษาไทย

In [ ]:
order = {
    "customer": "สมชาย",
    "message": "สั่งของไป 3 วันแล้วยังไม่ได้รับ เลขพัสดุก็เช็คไม่ได้ ขอคืนเงินได้ไหมครับ",
    "order_total_thb": 1290,
    "days_since_order": 3,
}
res = systemone(
    json.dumps(order, ensure_ascii=False),
    {
        "intent": {
            "type": "choice",
            "instructions": "What does the customer want?",
            "criteria": {
                "refund": "Wants money back",
                "tracking": "Wants to know where the parcel is",
                "cancel": "Wants to cancel the order",
                "other": "Something else",
            },
        },
        "angry": {"type": "noul", "instructions": "Is the customer upset?"},
        "sentiment": {"type": "score", "criteria": ["Very negative", "Negative", "Neutral", "Positive"]},
    },
)
show(res)

## 4. รูปภาพ (URL หรือ base64 data URI)

In [ ]:
from IPython.display import Image as IPImage, display

IMG = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
display(IPImage(url=IMG, width=360))
res = systemone(
    "Look at the image.",
    {
        "subject": {
            "type": "choice",
            "instructions": "What is the main subject of the image?",
            "criteria": {"dog": "A dog", "cat": "A cat", "car": "A car", "food": "Food"},
        },
        "outdoors": {"type": "noul", "instructions": "Was the photo taken outdoors?"},
    },
    images=[IMG],
)
show(res)

In [ ]:
# ส่งรูปจากไฟล์ในเครื่องเป็น base64
from PIL import Image, ImageDraw
img = Image.new("RGB", (320, 200), "white")
ImageDraw.Draw(img).rectangle([60, 40, 260, 160], fill="red")
buf = io.BytesIO(); img.save(buf, format="PNG")
display(img)
data_uri = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
res = systemone(
    "A generated test picture.",
    {"color": {"type": "choice", "instructions": "Main color of the shape?",
               "criteria": {"red": "Red", "green": "Green", "blue": "Blue"}}},
    images=[data_uri],
)
show(res)

## 5. ทดสอบ error handling

In [ ]:
bad_key = requests.post(f"{BASE}/v1/systemone", json={"model": "clef", "state": "x", "questions": {"q": {"type": "noul"}}},
                        headers={"Authorization": "Bearer wrong"}, timeout=30)
print("wrong key  ->", bad_key.status_code, bad_key.json())

no_q = requests.post(f"{BASE}/v1/systemone", json={"model": "clef", "state": "x", "questions": {}}, headers=HEADERS, timeout=30)
print("no question ->", no_q.status_code, no_q.json())

bad_type = requests.post(f"{BASE}/v1/systemone", json={"model": "clef", "state": "x", "questions": {"q": {"type": "yesno"}}},
                         headers=HEADERS, timeout=30)
print("bad type   ->", bad_type.status_code, bad_type.json())

## 6. Latency benchmark (batch 1 ทีละ request)

In [ ]:
Q = {"outage": {"type": "noul", "instructions": "Is a service down?"}}
systemone("warmup", Q)
lat, rtt = [], []
for i in range(20):
    r = systemone(f"Ticket #{i}: the website shows error 500 for every user.", Q)
    lat.append(r["latency_ms"]); rtt.append(r["rtt_ms"])
q = lambda xs, p: sorted(xs)[int(len(xs) * p) - 1]
print(f"server latency  median {statistics.median(lat):.1f} ms  p95 {q(lat, .95):.1f} ms")
print(f"round-trip      median {statistics.median(rtt):.1f} ms  p95 {q(rtt, .95):.1f} ms")

## 7. Concurrent requests (server serialize ทีละ request ด้วย lock)

In [ ]:
def one(i):
    return systemone(f"Customer {i}: I was charged twice for my subscription.", {
        "department": {"type": "choice", "instructions": "Which team should handle the message?",
                       "criteria": {"billing": "Payments, charges or invoices", "technical": "Bugs or outages"}}})

t0 = time.perf_counter()
with ThreadPoolExecutor(8) as ex:
    results = list(ex.map(one, range(16)))
wall = time.perf_counter() - t0
print(f"16 requests / 8 threads: {wall:.2f} s  -> {16 / wall:.1f} req/s")
from collections import Counter
print(Counter(r["answers"]["department"]["choice"] for r in results))